# 04 · Roll back one pair

Use this when a run is (partly) done and one pair must be produced again, e.g. after a config fix,
without starting a new run. It deletes that pair's *signals* (the objects that say "this step is
done"), so the next run of notebook 01/02 redoes exactly that part. The dry run resolves every
logical key to its exact absolute path or S3 URI; apply verifies each target is absent afterward.

| `STAGE` | deletes (for `PAIR`) | then run |
| --- | --- | --- |
| `dead` | `dead/` units | 02 (the Right side retries them), 03 |
| `col` | `_INDEX/`, `units/`, `results/`, `dead/`, `manifest/<pair>.keys.json`, `manifest/_CLOSED`, `_STATUS.json` | 02, 03 |
| `row` | all of `col` + `row_check/<pair>.json`, `row_check/<pair>.left.json`, both `_CLOSED`, `manifest/<pair>.json`, `reports/<pair>_compare_row.csv` | 01, 02, 03 |

Which stage a config change needs:

| Changed in the config | Stage |
| --- | --- |
| `col_map`, `col_type_overrides`, `sample`, `key`, `key_families`, `strict`, `vintage` | `col` with `REBUILD_MANIFEST = True` (no recount) |
| anything in `left` / `right` (`date_col`, `date_type`, `table`, `where`, ...), `fromDate`, `toDate`, `date_synonyms` | `row` |
| nothing (dead units, a crashed Stage 2) | `dead` or `col` |

A Stage 1 change with `STAGE = "col"` is refused. When several pairs or most settings changed,
start a new run instead (notebook 00 with `RUN_ID = None`). The same tool on the command line:
`dtrack tools rollback --run <run folder or s3://.../runs/<run_id>> --pair P --stage col [--config c.json] [--yes]`.

In [ ]:
# Parameters: keep CONFIG / OUT identical in notebooks 00-03
CONFIG = "c.json"  # pairs config (col_map files resolve next to it)
OUT = ".dtrack-local/runs"  # local runs root; one folder per run id
RUN_ID = None  # None: 00 starts a new run, 01-03 continue the run 00 opened (OUT/LATEST)
DISPOSITION = (
    "external_links"  # "inline" when presigned downloads are blocked (<= 25 MiB per window)
)
PUSH_SAMPLE = True  # False only when you need to land the full Right window locally
PAIRS = None  # e.g. ["orders"]; None = every active pair
PAIR = "orders"  # the pair to roll back
STAGE = "col"  # dead | col | row
REBUILD_MANIFEST = True  # STAGE col: rebuild manifest/<PAIR>.json from CONFIG
APPLY = False  # False: show the plan only
REGENERATE = False  # True: redo the stage here instead of re-running 01/02/03

In [ ]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)
pd.set_option("display.max_columns", 40, "display.width", 200)

from dtrack_local import pipeline

In [ ]:
RUN_ID = RUN_ID or (Path(OUT) / "LATEST").read_text().strip()
run = pipeline.open_run(
    CONFIG, run_id=RUN_ID, out=OUT, disposition=DISPOSITION, push_sample=PUSH_SAMPLE, pairs=PAIRS
)
print("run_id:", run.run_id)
print("folder:", run.folder.resolve())
print("status:", run.status())

In [ ]:
from dtrack_left.checks import rollback
from dtrack_left.cli.config import load_config

config = load_config(CONFIG) if STAGE == "col" and REBUILD_MANIFEST else None
outcome = rollback.rollback(run.store, PAIR, STAGE, config=config, apply=APPLY)
print(
    f"{outcome['planned']} object(s) {'deleted' if APPLY else 'would be deleted (APPLY = False)'}"
)
display(pd.DataFrame(outcome["deletions"] if APPLY else outcome["targets"]))
if APPLY:
    print({name: outcome[name] for name in ("deleted", "missing", "failed")})
print("next:", *outcome["next"], sep="\n  ")

## Regenerate here (optional)
The same calls notebooks 01-03 make, limited to `PAIR`.

In [ ]:
if APPLY and REGENERATE and not outcome["failed"]:
    run = pipeline.open_run(
        CONFIG,
        run_id=run.run_id,
        out=OUT,
        disposition=DISPOSITION,
        push_sample=PUSH_SAMPLE,
        pairs=[PAIR],
    )
    if STAGE == "row":
        print(pipeline.row_check(run))
    print(pipeline.col_check(run))
    summary = pipeline.report(run)["summary"]
    print(
        json.dumps(
            {k: summary[k] for k in ("status", "findings", "units_expected", "units_completed")},
            indent=2,
        )
    )
else:
    print(
        "set APPLY = True and REGENERATE = True to redo the stage here, or re-run notebooks 01/02/03 with PAIRS = [PAIR]"
    )